# Setup
Notebooks call reusable functions from `src/`.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.config import load_config, resolve_paths, set_global_seed, get_seed
config = load_config(ROOT / 'configs/project_config.yaml')
paths = resolve_paths(config)
set_global_seed(get_seed(config))
print('project root:', paths.root)


## SHAP analysis

In [ ]:
import joblib, numpy as np
from src.data_loader import load_parquet
from src.feature_engineering import get_model_feature_columns
from src.temporal_split import chronological_date_split, masks_from_split
from src.explainability import run_shap_analysis
df = load_parquet(paths.processed).sort_values('timestamp').reset_index(drop=True)
bundle = joblib.load(paths.models/'xgboost_model.joblib')
split = chronological_date_split(df, 0.6, 0.2, 0.2, 60)
masks = masks_from_split(len(df), split)
test_idx = np.where(masks['test'] & df['label'].notna().to_numpy())[0]
# Load predictions if available via recompute
from src.train_logistic import prepare_xy
X,y,idx = prepare_xy(df, bundle['feature_cols'], masks['test'] & df['label'].notna().to_numpy())
pred = {'index': idx.to_numpy(), 'y_true': y, 'y_pred': bundle['model'].predict(X), 'y_proba': bundle['model'].predict_proba(X)}
out = run_shap_analysis(bundle['model'], df, bundle['feature_cols'], test_idx, pred, config, paths)
print(out['top_features'][:5])
